In [12]:
import requests

base_url = "http://localhost:8002"
endpoint = f"{base_url}/api/events/"
print(endpoint)

http://localhost:8002/api/events/


In [13]:
# health check first
r = requests.get(f"{base_url}/healthz")
print(r.status_code, r.json())

200 {'status': 'ok'}


In [14]:
# create - note the response has `time` now, not `created_at`
r = requests.post(endpoint, json={"page": "/about", "description": "first event"})
print(r.status_code)
created = r.json()
created

201


{'id': 15,
 'description': 'first event',
 'time': '2026-09-12T07:23:12.147376Z',
 'updated_at': '2026-09-12T07:23:12.147387Z',
 'page': '/about'}

In [15]:
event_id = created["id"]
print("event_id:", event_id)
print("time:", created["time"])
print("updated_at:", created["updated_at"])

event_id: 15
time: 2026-09-12T07:23:12.147376Z
updated_at: 2026-09-12T07:23:12.147387Z


In [16]:
# seed some traffic so the buckets have something to count
pages = ["/about", "/about", "/pricing", "/pricing", "/pricing", "/blog"]
for page in pages:
    requests.post(endpoint, json={"page": page})

print("posted", len(pages))

posted 6


In [17]:
# GET / is an aggregation now - one row per (bucket, page), not raw events
r = requests.get(endpoint)
rows = r.json()
print(r.status_code, "rows:", len(rows))

for row in rows:
    print(f'{row["bucket"]}  {row["page"]:<12} {row["count"]}')

200 rows: 3
2026-09-12T00:00:00Z  /about       8
2026-09-12T00:00:00Z  /blog        3
2026-09-12T00:00:00Z  /pricing     8


In [18]:
# smaller buckets - same events, finer time resolution
r = requests.get(endpoint, params={"duration": "1 hour"})
for row in r.json():
    print(f'{row["bucket"]}  {row["page"]:<12} {row["count"]}')

2026-09-12T07:00:00Z  /blog        3
2026-09-12T07:00:00Z  /pricing     8
2026-09-12T07:00:00Z  /about       8


In [19]:
# filter to specific pages (repeated query param)
r = requests.get(endpoint, params={"pages": ["/about", "/blog"]})
for row in r.json():
    print(f'{row["bucket"]}  {row["page"]:<12} {row["count"]}')

2026-09-12T00:00:00Z  /about       8
2026-09-12T00:00:00Z  /blog        3


In [20]:
# detail
r = requests.get(f"{endpoint}{event_id}")
print(r.status_code)
r.json()

200


{'id': 15,
 'description': 'first event',
 'time': '2026-09-12T07:23:12.147376Z',
 'updated_at': '2026-09-12T07:23:12.147387Z',
 'page': '/about'}

In [21]:
# missing id should 404
r = requests.get(f"{endpoint}999999")
print(r.status_code, r.json())

404 {'detail': 'Event not found'}


In [22]:
# update - updated_at should move, time should not
r = requests.put(f"{endpoint}{event_id}", json={"description": "updated!"})
print(r.status_code)
updated = r.json()

print("description:", updated["description"])
print("time same:", updated["time"] == created["time"])
print("updated_at changed:", updated["updated_at"] != created["updated_at"])

200
description: updated!
time same: True
updated_at changed: True


In [23]:
# delete returns 204 with no body
r = requests.delete(f"{endpoint}{event_id}")
print(r.status_code, repr(r.text))

204 ''


In [24]:
# gone now
r = requests.get(f"{endpoint}{event_id}")
print(r.status_code, r.json())

404 {'detail': 'Event not found'}
